In [1]:
# -----------------------------------------------------------------------------
# NOTEBOOK EXECUTION CONTROL
# -----------------------------------------------------------------------------
# Set this to True to run individual component tests sequentially
RUN_DEBUG_STEPS = True


In [2]:
import sys
import os
import logging
import json
import dataclasses

# -----------------------------------------------------------------------------
# PATH SETUP: Point to your project root to load 'common' and 'microservices'
# -----------------------------------------------------------------------------
# If this notebook is in a 'notebooks' subfolder, use '..'
# If in root, use '.'
project_root = os.path.abspath('../../..') 
if project_root not in sys.path:
    sys.path.append(project_root)

# -----------------------------------------------------------------------------
# IMPORT EXISTING BACKEND DATA STRUCTURES
# -----------------------------------------------------------------------------
try:
    from common.models.api.redis_models import (
        Article, NLPResult, NLPOptions, Claim, Entity, SentenceScore, BiasProfile
    )
    from microservices.nlp.models.base import NLPComponent
    print("Successfully loaded backend data structures.")
except ImportError as e:
    print(f"Import Failed: {e}")
    print("Ensure 'project_root' correctly points to the folder containing 'common/' and 'microservices/'")

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("Notebook")

Successfully loaded backend data structures.


In [3]:
import json
import os

# 1. Load data from article.json (populated from the RTF content)
# We assume article.json is in the same directory as this notebook
json_path = 'article3.json' 

if not os.path.exists(json_path):
    print(f"Error: '{json_path}' not found in the current directory.")
    # Stop execution if file is missing
    raise FileNotFoundError(f"Please ensure {json_path} is next to this notebook.")

with open(json_path, 'r') as f:
    data = json.load(f)
    print(f"Successfully loaded '{json_path}'")

# 2. Create Article Object (using backend model)
# We map keys from the updated JSON format to the Article attributes
article = Article(
    title=data.get('article_title', 'Unknown Title'),
    text=data.get('article_text', ''),
    # Map 'article_url' from JSON to the Article's 'link' field
    link=data.get('article_url', ''),
    summary=data.get('article_summary', '')
)

# 3. Initialize Result and Options
result = NLPResult()
options = NLPOptions(min_confidence=0.8)

print(f"Initialized Article: {article.title}")

Successfully loaded 'article3.json'
Initialized Article: FBI raids Georgia election office over 2020 voter fraud claims


## Preprocessor

In [4]:
import logging
import spacy
import re
from typing import List

# Local imports
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore

logger = logging.getLogger(__name__)

class Preprocessor(NLPComponent):
    """
    "Universal Janitor" Preprocessor.
    
    Strategies applied:
    1. Regex Cleaning: Removes distinct artifacts (Dates, UI buttons, Footers) using strict patterns.
    2. Footer Cutoff: Stops processing the text entirely once footer keywords are detected.
    3. Linguistic Filtering: Uses Spacy's POS tagger to remove short lines (< 7 tokens) 
       that look like sentences but lack verbs (e.g., "Politics", "Frank Gardner").

    Returns sentences via a local list; does NOT write to result.sentences.
    """
    def __init__(self):
        logger.info("Preprocessor: Loading Spacy 'en_core_web_sm' model...")
        try:
            self.nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
        except OSError:
            logger.error("Spacy model not found. Run: python -m spacy download en_core_web_sm")
            raise

    def _clean_and_repair_structure(self, raw_text: str) -> str:
        if not raw_text: return ""

        lines = raw_text.split('\n')
        cleaned_lines = []
        
        footer_cutoff_pattern = re.compile(r'(?i)^('
            r'more from (the )?bbc|related (content|stories|topics)|up next|most popular|'
            r'have you read\?|more on geographies|license and republishing|content index|'
            r'bbc\.com help|privacy policy|about us|follow .* on|sign up for'
        r')')
        
        time_meta_pattern = re.compile(r'(?i)^('
            r'\d+\s+(hour|minute|day|second|hr|min)s?\s+ago|'
            r'updated\s+.*|'
            r'\d+\s+min\s+read|'
            r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+\d{1,2},?\s+\d{4}'
        r')')

        credits_pattern = re.compile(r'(?i)^('
            r'(image|photo|source|graphic|credits?):|'
            r'(left|right|top|bottom):|'
            r'analysis by|'
            r'this article is part of:|'
            r'unknown\.|'
            r'getty images|epa|afp|ugc|reuters|ap|copyright|davidoff studios'
        r')')

        ui_pattern = re.compile(r'(?i)^('
            r'register|sign in|log in|'
            r'skip to.*|'
            r'share|save|follow|subscribe|'
            r'menu|home|news|sport|weather|'
            r'listen to .* read this article|'
            r'loading\.\.\.|'
            r'create a free account|'
            r'terms of use'
        r')')

        byline_pattern = re.compile(r'(?i)^('
            r'by\s+[A-Z][a-z]+\s+[A-Z][a-z]+|'
            r'.*correspondent.*|'
            r'writer,.*'
        r')')

        for line in lines:
            line = line.strip()
            if not line: continue 
            if len(line) < 4: continue
            if footer_cutoff_pattern.search(line): break
            if ui_pattern.search(line): continue
            if time_meta_pattern.search(line): continue
            if credits_pattern.search(line): continue
            if byline_pattern.search(line) and len(line) < 50: continue
            if line[-1] not in ".?!:;\"'":
                line += "."
            cleaned_lines.append(line)
            
        return " ".join(cleaned_lines)

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> List[SentenceScore]:
        """
        Cleans and tokenizes the article text.
        Returns a local list of SentenceScore objects (does NOT write to result).
        """
        raw_text = getattr(article, 'text', getattr(article, 'content', ""))
        clean_text = self._clean_and_repair_structure(raw_text)
        
        if not clean_text:
            logger.warning("Preprocessor: Text was empty after cleaning.")
            return []

        doc = self.nlp(clean_text)
        
        sentence_objects = []
        for idx, span in enumerate(doc.sents):
            text_segment = span.text.strip()
            token_count = len(span)
            
            if token_count < 7:
                if "?" in text_segment:
                    pass 
                else:
                    has_verb = any(token.pos_ in ["VERB", "AUX"] for token in span)
                    if not has_verb:
                        continue

            s_obj = SentenceScore(
                index=idx, 
                text=text_segment, 
                score=0.0, 
                embedding=None
            )
            sentence_objects.append(s_obj)

        logger.info(f"Preprocessor: Cleaned & Split. Result: {len(sentence_objects)} sentences.")
        return sentence_objects

if RUN_DEBUG_STEPS:
    print("--- Running Preprocessor Test ---")
    try:
        pre = Preprocessor()
        sentences = pre.run(article, result, options)
        
        print(f"Success! Split into {len(sentences)} sentences.")
        for s in sentences:
            print(f"  [{s.index}] {s.text}")
            
    except Exception as e:
        print(f"Error: {e}")


__main__ - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


--- Running Preprocessor Test ---


__main__ - INFO - Preprocessor: Cleaned & Split. Result: 26 sentences.


Success! Split into 26 sentences.
  [0] FBI raids Georgia election office over 2020 voter fraud claims.
  [1] The FBI raided a Georgia election office on Wednesday as it examined allegations of voter fraud in the 2020 US election.
  [2] In a statement to Reuters, the FBI said it was conducting a "court-authorised law enforcement activity" at the Fulton County Election Hub.
  [3] Fulton County officials said that the government's warrant "sought a number of records related to 2020 elections".
  [4] Donald Trump lost the state and the county - the most populated in Georgia - to Joe Biden in the 2020 election and has long said his loss was due to fraud, a claim that is unsubstantiated.
  [5] The Department of Justice (DOJ) sued Fulton County officials in December, seeking election-related materials from 2020.
  [6] The FBI and Fulton County did not immediately respond to the BBC's requests for comment.
  [7] Agents with FBI vests were seen entering and exiting the election office on Wedne

## Entity Recognizer

In [5]:
import logging
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from typing import List, Dict, Any

# Local imports
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, Entity, SentenceScore

logger = logging.getLogger(__name__)

class EntityRecognizer(NLPComponent):
    """
    Named Entity Recognition using 'dslim/bert-base-NER-uncased'.

    Accepts a local sentences list and writes deduplicated entities
    into result.entities_in_article only.
    """
    BATCH_SIZE = 16

    def __init__(self):
        self.model_name = "dslim/bert-base-NER-uncased"
        self.device = 0 if torch.cuda.is_available() else -1
        use_fp16 = torch.cuda.is_available()

        logger.info(f"EntityRecognizer: Loading '{self.model_name}' "
                    f"on {'CUDA' if self.device == 0 else 'CPU'} "
                    f"(fp16={use_fp16})...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForTokenClassification.from_pretrained(
                self.model_name,
                torch_dtype=torch.float16 if use_fp16 else torch.float32,
            )
            self.nlp_pipeline = pipeline(
                "ner",
                model=self.model,
                tokenizer=self.tokenizer,
                aggregation_strategy="simple",
                device=self.device,
                batch_size=self.BATCH_SIZE,
            )
        except Exception as e:
            logger.error(f"EntityRecognizer: Failed to load model: {e}")
            raise

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> None:
        """
        Runs NER over all sentences.
        Writes deduplicated entities to result.entities_in_article.
        Does NOT modify the sentences list.
        """
        if not sentences:
            return

        # Filter out empty/whitespace-only texts — HF pipeline raises
        # "At least one input is required." if any empty string is passed.
        texts: List[str] = [s.text for s in sentences if s.text and s.text.strip()]
        if not texts:
            logger.warning("EntityRecognizer: All sentence texts are empty, skipping.")
            return

        logger.info(f"EntityRecognizer: Running NER on {len(texts)} sentences.")

        unique_entities: Dict[tuple, Dict[str, Any]] = {}

        try:
            # Pass plain Python list directly — avoids Arrow-backed Dataset issues
            for sent_results in self.nlp_pipeline(texts):
                for item in sent_results:
                    text = item["word"].strip()
                    label = item["entity_group"]
                    score = float(item["score"])

                    if len(text) < 3:
                        continue

                    key = (text.lower(), label)
                    if key not in unique_entities or score > unique_entities[key]["score"]:
                        e_obj = Entity(
                            entity_text=text,
                            type_of_entity=label,
                            start_char=item.get("start", 0),
                            end_char=item.get("end", 0),
                        )
                        unique_entities[key] = {"entity_obj": e_obj, "score": score}

            result.entities_in_article = [v["entity_obj"] for v in unique_entities.values()]
            logger.info(f"EntityRecognizer: Found {len(result.entities_in_article)} unique entities.")

        except Exception as e:
            logger.error(f"EntityRecognizer failed: {e}")
            raise

if RUN_DEBUG_STEPS:
    print("--- Running Entity Recognizer Test ---")
    print(f"Input: {len(sentences)} sentences from Preprocessor")
    try:
        ner = EntityRecognizer()
        ner.run(article, result, options, sentences)

        print(f"Found {len(result.entities_in_article)} unique entities.")
        for e in result.entities_in_article:
            print(f"  - {e.entity_text} ({e.type_of_entity})")

    except Exception as e:
        import traceback
        print(f"Error: {e}")
        traceback.print_exc()


/home/farhan/miniconda2/envs/nlp311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
__main__ - INFO - EntityRecognizer: Loading 'dslim/bert-base-NER-uncased' on CUDA (fp16=True)...


--- Running Entity Recognizer Test ---
Input: 26 sentences from Preprocessor


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 140.83it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER-uncased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
__main__ - INFO - EntityRecognizer: Running NER on 26 sentences.
__main__ - INFO - EntityRecognizer: Found 20 unique entities.


Found 20 unique entities.
  - fbi (ORG)
  - georgia (LOC)
  - reuters (ORG)
  - fulton county (LOC)
  - fulton county (ORG)
  - donald trump (PER)
  - joe biden (PER)
  - department of justice (ORG)
  - doj (ORG)
  - bbc (ORG)
  - mo ivory (PER)
  - democrat (MISC)
  - bill clinton (PER)
  - biden (PER)
  - trump (PER)
  - raffen (PER)
  - ##sperger (ORG)
  - white house (LOC)
  - trump (ORG)
  - georgian (MISC)


## Sentence Extraction + Deduplication

In [6]:
import logging
import torch
import numpy as np
import torch.nn.functional as F
from typing import List
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from torch.amp import autocast
from datasets import Dataset

# Local imports
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore

logger = logging.getLogger(__name__)

class SentenceExtraction(NLPComponent):
    """
    MODULAR SENTENCE EXTRACTION LAYER
    Combines extractive importance (BertSum-style CLS scoring) with
    logical deduplication (NLI cross-encoder).

    Accepts and returns a local sentences list; does NOT touch result.
    """
    SCORING_BATCH_SIZE = 16
    NLI_MAX_PAIRS      = 32

    def __init__(self, use_fp16: bool = True, default_top_k: int = 10):
        self.use_fp16 = use_fp16 and torch.cuda.is_available()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.default_top_k = default_top_k

        logger.info(f"SentenceExtraction: Initializing models on {self.device} "
                    f"(fp16={self.use_fp16})...")

        self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
        self.scoring_model = AutoModel.from_pretrained("bert-base-uncased").to(self.device)

        self.nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-distilroberta-base")
        self.nli_model = (
            AutoModelForSequenceClassification
            .from_pretrained("cross-encoder/nli-distilroberta-base")
            .to(self.device)
        )

        if self.use_fp16:
            self.scoring_model = self.scoring_model.half()
            self.nli_model = self.nli_model.half()

    def _get_salience_scores(self, sentences: List[str]) -> np.ndarray:
        scores = []
        with torch.no_grad():
            with autocast("cuda", enabled=self.use_fp16):
                for start in range(0, len(sentences), self.SCORING_BATCH_SIZE):
                    batch_texts = sentences[start : start + self.SCORING_BATCH_SIZE]
                    inputs = self.tokenizer(
                        batch_texts,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=512,
                    ).to(self.device)
                    outputs = self.scoring_model(**inputs)
                    cls_vecs = outputs.last_hidden_state[:, 0, :]
                    batch_scores = torch.mean(torch.abs(cls_vecs), dim=-1)
                    scores.extend(batch_scores.cpu().float().tolist())

        scores = np.array(scores)
        return (
            (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
            if len(scores) > 1
            else scores
        )

    def _is_redundant(self, candidate: str, already_selected: List[str]) -> bool:
        if not already_selected:
            return False

        pairs = [[prev, candidate] for prev in already_selected]
        with torch.no_grad():
            with autocast("cuda", enabled=self.use_fp16):
                inputs = self.nli_tokenizer(
                    pairs,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=512,
                ).to(self.device)
                outputs = self.nli_model(**inputs)
                probs = F.softmax(outputs.logits.float(), dim=-1)
                return bool((probs[:, 1] > 0.70).any())

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> List[SentenceScore]:
        """
        Scores and deduplicates sentences.
        Returns a filtered local list; does NOT modify result.
        """
        if not sentences:
            return []

        raw_texts = [s.text for s in sentences]
        scores = self._get_salience_scores(raw_texts)

        ranked_indices = np.argsort(scores)[::-1]
        selected_indices: List[int] = []
        selected_texts:   List[str] = []

        limit = getattr(options, "max_claims", self.default_top_k)

        for idx in ranked_indices:
            if len(selected_indices) >= limit:
                break
            candidate = raw_texts[idx]
            if not self._is_redundant(candidate, selected_texts):
                selected_indices.append(int(idx))
                selected_texts.append(candidate)
                sentences[idx].score = float(scores[idx])

        extracted = [sentences[i] for i in sorted(selected_indices)]
        logger.info(f"SentenceExtraction: Extracted {len(extracted)} salient, unique sentences.")
        return extracted


In [7]:
if RUN_DEBUG_STEPS:
    print("--- Running Sentence Extraction Test ---")
    try:
        extractor = SentenceExtraction(use_fp16=True)
        sentences = extractor.run(article, result, options, sentences)
        
        print(f"Success! Extracted {len(sentences)} sentences.")
        print("Top 5 Extracted Sentences:")
        for i, s in enumerate(sentences[:5]):
            print(f"  [{s.index}] (Score: {s.score:.4f}) {s.text[:100]}...")
            
    except Exception as e:
        print(f"Error: {e}")


__main__ - INFO - SentenceExtraction: Initializing models on cuda (fp16=True)...


--- Running Sentence Extraction Test ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 304.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 247.72it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]   

Success! Extracted 10 sentences.
Top 5 Extracted Sentences:
  [0] (Score: 1.0000) FBI raids Georgia election office over 2020 voter fraud claims....
  [1] (Score: 0.6579) The FBI raided a Georgia election office on Wednesday as it examined allegations of voter fraud in t...
  [9] (Score: 0.5770) "This is an assault on your vote," Fulton County Commissioner Mo Ivory said at a press conference ou...
  [11] (Score: 0.7715) The 2020 election marked the first time since 1992 that a Democrat had won the southern US state of ...
  [17] (Score: 0.6241) The state of Georgia, and Fulton County in particular, were a major focus of Trump's efforts to over...


## Decontextualizer

In [8]:
import torch
import logging
import spacy
import re
from datasets import Dataset
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from typing import List, Optional

# Local imports
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore

logger = logging.getLogger(__name__)

class Decontextualizer(NLPComponent):
    """
    MODULAR DECONTEXTUALIZER LAYER

    Accepts and returns a local sentences list; does NOT touch result.
    """
    QA_BATCH_SIZE  = 8
    GEN_BATCH_SIZE = 8
    BM25_TOP_K     = 3

    _LABEL_RE = re.compile(
        r"^(false|true|fake|entailment|neutral|contradiction)\b[\s:,]*",
        flags=re.IGNORECASE,
    )

    def __init__(self, use_gpu: bool = True):
        self.device    = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
        self.device_id = 0 if self.device == "cuda" else -1
        use_fp16       = self.device == "cuda"

        logger.info(f"Decontextualizer: Initializing on {self.device} (fp16={use_fp16})...")

        try:
            self.nlp = spacy.load("en_core_web_sm")
        except OSError:
            logger.error("Run: python -m spacy download en_core_web_sm")
            raise

        self.qg_tokenizer = AutoTokenizer.from_pretrained(
            "mrm8488/t5-base-finetuned-question-generation-ap"
        )
        self.qg_model = AutoModelForSeq2SeqLM.from_pretrained(
            "mrm8488/t5-base-finetuned-question-generation-ap",
            torch_dtype=torch.float16 if use_fp16 else torch.float32,
        ).to(self.device)

        self.qa_pipe = pipeline(
            "question-answering",
            model="deepset/roberta-base-squad2",
            device=self.device_id,
            batch_size=self.QA_BATCH_SIZE,
            torch_dtype=torch.float16 if use_fp16 else torch.float32,
        )

        self.gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(
            "google/flan-t5-base",
            torch_dtype=torch.float16 if use_fp16 else torch.float32,
        ).to(self.device)

    def _sanitize(self, text: str) -> str:
        return self._LABEL_RE.sub("", text).strip()

    def _generate_batch(self, prompts, tokenizer, model, max_length: int = 128) -> list[str]:
        if not prompts:
            return []
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=512,
        ).to(self.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_length=max_length, num_beams=4,
                repetition_penalty=2.5, early_stopping=True,
            )
        return [self._sanitize(tokenizer.decode(o, skip_special_tokens=True)) for o in outputs]

    def _extract_units(self, doc) -> list[str]:
        units: list[str] = []
        for ent in doc.ents:
            units.append(ent.text)
        for token in doc:
            if token.pos_ == "PRON":
                units.append(token.text)
        for chunk in doc.noun_chunks:
            if len(chunk.text.split()) < 5:
                units.append(chunk.text)
        for token in doc:
            if token.pos_ == "VERB" and token.dep_ in ("ROOT", "relcl", "advcl", "xcomp"):
                phrase_tokens = [token] + [
                    c for c in token.children
                    if c.dep_ in ("dobj", "prt", "attr") and c.i < token.i + 4
                ]
                phrase_tokens.sort(key=lambda t: t.i)
                phrase = " ".join(t.text for t in phrase_tokens)
                if phrase:
                    units.append(phrase)
        return [u for u in set(units) if len(u.split()) < 6 and len(u) > 1]

    def _bm25_retrieve(self, query: str, doc_sentences: list[str]) -> str:
        if not doc_sentences:
            return ""
        tokenized_corpus = [s.lower().split() for s in doc_sentences]
        bm25 = BM25Okapi(tokenized_corpus)
        scores = bm25.get_scores(query.lower().split())
        top_k = min(self.BM25_TOP_K, len(doc_sentences))
        top_indices = scores.argsort()[-top_k:][::-1]
        return " ".join(doc_sentences[i] for i in sorted(top_indices))

    def _process_sentence(self, sent_text: str, units: list[str],
                          doc_sentences: list[str]) -> Optional[str]:
        if not units:
            return None

        qg_prompts = [f"answer: {u} context: {sent_text}" for u in units]
        questions  = self._generate_batch(qg_prompts, self.qg_tokenizer, self.qg_model)

        qa_inputs = []
        for question in questions:
            evidence = self._bm25_retrieve(question, doc_sentences)
            qa_inputs.append({"question": question, "context": evidence or sent_text})

        qa_ds = Dataset.from_list(qa_inputs)
        try:
            qa_results = list(self.qa_pipe(
                qa_ds["question"], qa_ds["context"], batch_size=self.QA_BATCH_SIZE,
            ))
        except Exception as e:
            logger.warning(f"Decontextualizer: QA batch failed — {e}")
            return None

        qa2d_prompts = [
            f"Convert to a declarative sentence: Q: {q} A: {r['answer']}"
            for q, r in zip(questions, qa_results)
            if r["score"] > 0.35
        ]

        if not qa2d_prompts:
            return None

        try:
            context_sentences = self._generate_batch(qa2d_prompts, self.gen_tokenizer, self.gen_model)
        except Exception as e:
            logger.warning(f"Decontextualizer: QA2D batch failed — {e}")
            return None

        return " ".join(s for s in context_sentences if s) or None

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> List[SentenceScore]:
        """
        Rewrites each sentence to be self-contained.
        Returns updated local sentences list; does NOT modify result.
        """
        if not sentences:
            return []

        doc_sentences = [
            s.strip()
            for s in re.split(r"(?<=[.!?])\s+", article.text)
            if len(s.strip()) > 10
        ]

        for sent_obj in sentences:
            cleaned_text = self._sanitize(sent_obj.text)
            if not cleaned_text:
                sent_obj.text = ""
                continue

            doc            = self.nlp(cleaned_text)
            units          = self._extract_units(doc)
            filtered_units = [u for u in units if len(u.split()) < 6][:6]
            full_context   = self._process_sentence(cleaned_text, filtered_units, doc_sentences)

            if full_context:
                final_prompt = (
                    f"Rewrite the sentence to be self-contained by incorporating "
                    f"specific details from the context.\n"
                    f"Context: {full_context}\n"
                    f"Sentence: {cleaned_text}\n"
                    f"Rewrite:"
                )
                try:
                    rewrites = self._generate_batch(
                        [final_prompt], self.gen_tokenizer, self.gen_model, max_length=128
                    )
                    sent_obj.text = rewrites[0] if rewrites else cleaned_text
                except Exception as e:
                    logger.warning(f"Decontextualizer: Final rewrite failed — {e}")
                    sent_obj.text = cleaned_text
            else:
                sent_obj.text = cleaned_text

        logger.info("Decontextualizer: Complete.")
        return sentences


In [ ]:
if RUN_DEBUG_STEPS:
    print("--- Running Decontextualizer Test ---")
    try:
        # Rebuild a clean local sentences list from scratch
        clean_result  = NLPResult()
        clean_options = NLPOptions(min_confidence=0.8, max_claims=10)

        pre_tmp  = Preprocessor()
        sents_tmp = pre_tmp.run(article, clean_result, clean_options)

        ner_tmp  = EntityRecognizer()
        ner_tmp.run(article, clean_result, clean_options, sents_tmp)

        ext_tmp  = SentenceExtraction(use_fp16=True)
        sents_tmp = ext_tmp.run(article, clean_result, clean_options, sents_tmp)

        print(f"\n{len(sents_tmp)} sentences after extraction. Running Decontextualizer...")

        decon = Decontextualizer(use_gpu=True)
        old_texts  = [s.text for s in sents_tmp]
        sents_tmp  = decon.run(article, clean_result, clean_options, sents_tmp)

        print("\nDecontextualization Results:")
        for old, new_obj in zip(old_texts, sents_tmp):
            print(f"  Original : {old}")
            print(f"  Rewritten: {new_obj.text}")
            print("  " + "-" * 60)

    except Exception as e:
        import traceback
        print(f"Error: {e}")
        traceback.print_exc()


__main__ - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


--- Running Decontextualizer Test ---


__main__ - INFO - Preprocessor: Cleaned & Split. Result: 26 sentences.
__main__ - INFO - EntityRecognizer: Loading 'dslim/bert-base-NER-uncased' on CUDA (fp16=True)...
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 145.62it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER-uncased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
__main__ - INFO - EntityRecognizer: Running NER on 26 sentences.
__main__ - INFO - EntityRecognizer: Found 20 unique entities.
__main__ - INFO - SentenceExtraction: Initializing models on cuda (fp16=True)...
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 261.34it/s, Materializing param=pooler.dense.weight] 

## Checkworthy

In [ ]:
import logging
import spacy
from typing import List
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore

logger = logging.getLogger(__name__)

class CheckWorthiness(NLPComponent):
    """
    MODULAR CHECK-WORTHINESS LAYER

    Accepts and returns a local sentences list; does NOT touch result.
    Uses spaCy's nlp.pipe() for batched processing.
    """
    BATCH_SIZE = 32

    def __init__(self):
        logger.info("CheckWorthiness: Loading Spacy model...")
        # "textcat" removed — not present in en_core_web_sm
        self.nlp = spacy.load("en_core_web_sm", disable=["lemmatizer"])

        self.reporting_verbs = {
            "say", "claim", "state", "report", "announce",
            "confirm", "warn", "accuse",
        }
        self.speculative_keywords = {
            "could", "might", "may", "would", "predict", "expect", "poised",
            "likely", "potential", "possibly", "perhaps", "if", "future",
        }

    def _score_doc(self, doc) -> tuple[float, bool]:
        entities = [e for e in doc.ents if e.label_ in
                    ("PERSON", "ORG", "GPE", "EVENT", "FAC", "NORP")]
        numbers  = [e for e in doc.ents if e.label_ in
                    ("MONEY", "PERCENT", "CARDINAL", "DATE", "QUANTITY")]

        verbs = [t for t in doc if t.pos_ == "VERB"]
        has_reporting  = any(v.lemma_.lower() in self.reporting_verbs  for v in verbs)
        has_action     = any(v.lemma_.lower() not in self.reporting_verbs for v in verbs)
        is_speculative = any(t.text.lower() in self.speculative_keywords for t in doc)

        score = 0.0
        if len(entities) >= 1: score += 0.3
        if len(entities) >= 2: score += 0.1
        if len(numbers)  >= 1: score += 0.4
        if has_reporting:       score += 0.1
        if has_action:          score += 0.1
        if is_speculative:      score -= 0.5

        score = max(0.0, min(1.0, score))
        return score, score >= 0.6

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> List[SentenceScore]:
        """
        Scores each sentence for check-worthiness.
        Returns the same list with confidence/is_checkworthy populated.
        Does NOT modify result.
        """
        if not sentences:
            return []

        logger.info(f"CheckWorthiness: Evaluating {len(sentences)} sentences "
                    f"(batch_size={self.BATCH_SIZE})...")

        texts = [s.text for s in sentences]
        for s_obj, doc in zip(sentences, self.nlp.pipe(texts, batch_size=self.BATCH_SIZE)):
            score, is_checkworthy = self._score_doc(doc)
            s_obj.confidence     = float(score)
            s_obj.is_checkworthy = is_checkworthy

        logger.info("CheckWorthiness: Scoring complete.")
        return sentences


In [ ]:
if RUN_DEBUG_STEPS:
    print("--- Running Check-Worthiness Test ---")
    try:
        cw = CheckWorthiness()
        sentences = cw.run(article, result, options, sentences)
        
        print("\nCheck-Worthiness Scores:")
        for s in sentences:
            status = "[CHECK]" if s.is_checkworthy else "[IGNORE]"
            print(f"{status} Score: {s.confidence:.2f} | {s.text[:80]}...")

    except Exception as e:
        print(f"Error: {e}")


__main__ - INFO - CheckWorthiness: Loading Spacy model...


--- Running Check-Worthiness Test ---


__main__ - INFO - CheckWorthiness: Evaluating 10 sentences (batch_size=32)...
__main__ - INFO - CheckWorthiness: Scoring complete.



Check-Worthiness Scores:
[CHECK] Score: 0.90 | FBI raids Georgia election office over 2020 voter fraud claims....
[CHECK] Score: 0.90 | The FBI raided a Georgia election office on Wednesday as it examined allegations...
[IGNORE] Score: 0.50 | "This is an assault on your vote," Fulton County Commissioner Mo Ivory said at a...
[CHECK] Score: 0.90 | The 2020 election marked the first time since 1992 that a Democrat had won the s...
[CHECK] Score: 0.90 | The state of Georgia, and Fulton County in particular, were a major focus of Tru...
[IGNORE] Score: 0.40 | Numerous courts rejected legal challenges and claims made by Trump and his allie...
[IGNORE] Score: 0.50 | Raffensperger, whose office oversees Georgia's elections and certifies results, ...
[CHECK] Score: 0.90 | Trump faced two criminal indictments related to alleged election interference in...
[CHECK] Score: 0.90 | Trump officials sue Georgia county to force release of 2020 voting records....
[IGNORE] Score: 0.50 | WW1 toxic compou

In [ ]:
import logging
import torch
import numpy as np
from typing import List
from datasets import Dataset
from sentence_transformers import SentenceTransformer

# Local imports
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore

logger = logging.getLogger(__name__)

class Embedder(NLPComponent):
    """
    MODULAR EMBEDDER LAYER
    Model: 'sentence-transformers/all-mpnet-base-v2' (768-dim).

    Accepts and returns a local sentences list; does NOT touch result.
    """
    BATCH_SIZE = 32

    def __init__(self, model_name: str = "sentence-transformers/all-mpnet-base-v2"):
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        use_fp16 = torch.cuda.is_available()

        logger.info(f"Embedder: Loading {self.model_name} on {self.device} (fp16={use_fp16})...")
        try:
            self.model = SentenceTransformer(self.model_name, device=self.device)
            if use_fp16:
                self.model.half()
        except Exception as e:
            logger.error(f"Embedder: Failed to load model: {e}")
            raise

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> List[SentenceScore]:
        """
        Embeds each sentence in-place.
        Returns the same list with embedding fields populated.
        Does NOT modify result.
        """
        if not sentences:
            logger.info("Embedder: No sentences to process.")
            return []

        texts = [s.text for s in sentences]
        ds    = Dataset.from_dict({"text": texts})

        try:
            embeddings = self.model.encode(
                ds["text"],
                batch_size=self.BATCH_SIZE,
                show_progress_bar=len(texts) > self.BATCH_SIZE,
                convert_to_numpy=True,
                normalize_embeddings=False,
            )

            for i, sent in enumerate(sentences):
                sent.embedding = embeddings[i].tolist()

            logger.info(f"Embedder: Vectorized {len(texts)} sentences.")

        except Exception as e:
            logger.error(f"Embedder failed: {e}")
            raise

        return sentences


In [ ]:
import logging
import time
from typing import List
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, Claim, SentenceScore

logger = logging.getLogger(__name__)

class ClaimExtraction(NLPComponent):
    """
    THE ORCHESTRATOR
    Threads a local List[SentenceScore] through every stage.
    Only writes to result at the very end (claims_in_article, entities_in_article).
    """
    def __init__(self, use_gpu: bool = True):
        logger.info("ClaimExtraction: Initializing pipeline orchestrator...")
        self.preprocessor       = Preprocessor()
        self.entity_recognizer  = EntityRecognizer()
        self.sentence_extractor = SentenceExtraction()
        self.decontextualizer   = Decontextualizer(use_gpu=use_gpu)
        self.checkworthiness    = CheckWorthiness()
        self.final_embedder     = Embedder()

    def _map_entities_to_sentences(self, sentences: List[SentenceScore], result: NLPResult):
        """Maps global entities onto each sentence by text overlap."""
        if not sentences or not result.entities_in_article:
            return
        for s_obj in sentences:
            s_obj.entities = [
                ent for ent in result.entities_in_article
                if ent.entity_text.lower() in s_obj.text.lower()
            ]

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        total_start = time.time()

        # Stage 1 — Preprocessing → local list
        t = time.time()
        sentences = self.preprocessor.run(article, result, options)
        logger.info(f"Stage 1 [Preprocessor] complete in {time.time()-t:.2f}s — {len(sentences)} sentences")

        # Stage 2 — NER → writes to result.entities_in_article only
        t = time.time()
        self.entity_recognizer.run(article, result, options, sentences)
        logger.info(f"Stage 2 [NER] complete in {time.time()-t:.2f}s — {len(result.entities_in_article)} entities")

        # Stage 3 — Extraction + deduplication → filtered local list
        t = time.time()
        sentences = self.sentence_extractor.run(article, result, options, sentences)
        logger.info(f"Stage 3 [SentenceExtraction] complete in {time.time()-t:.2f}s — {len(sentences)} kept")

        # Stage 4 — Decontextualization → rewrites texts in local list
        t = time.time()
        sentences = self.decontextualizer.run(article, result, options, sentences)
        logger.info(f"Stage 4 [Decontextualizer] complete in {time.time()-t:.2f}s")

        # Stage 5 — Check-worthiness scoring
        t = time.time()
        sentences = self.checkworthiness.run(article, result, options, sentences)
        logger.info(f"Stage 5 [CheckWorthiness] complete in {time.time()-t:.2f}s")

        # Stage 5.5 — Map entities onto sentences
        t = time.time()
        self._map_entities_to_sentences(sentences, result)
        logger.info(f"Stage 5.5 [Entity Mapping] complete in {time.time()-t:.2f}s")

        # Stage 6 — Embed sentences
        t = time.time()
        sentences = self.final_embedder.run(article, result, options, sentences)
        logger.info(f"Stage 6 [Embedder] complete in {time.time()-t:.2f}s")

        # Stage 7 — Convert local SentenceScore list → result.claims_in_article
        t = time.time()
        result.claims_in_article = [
            Claim(
                confidence=s.confidence,
                source_sentence_indices=[s.index],
                decontextualised_claim_text=s.text,
                decontextualised_claim_embedding=s.embedding,
                NER_entities=getattr(s, "entities", []),
            )
            for s in sentences
        ]
        logger.info(
            f"Stage 7 [Sentence→Claim] complete in {time.time()-t:.2f}s — "
            f"{len(result.claims_in_article)} claims stored"
        )

        logger.info(f"--- Pipeline Finished in {time.time()-total_start:.2f}s ---")


In [ ]:
if RUN_DEBUG_STEPS:
    print("--- Running Embedder Test ---")
    try:
        emb = Embedder()
        sentences = emb.run(article, result, options, sentences)
        
        print(f"Successfully vectorized {len(sentences)} sentences.")
        if sentences:
            print(f"Sample Embedding (First 3 dims): {sentences[0].embedding[:3]}...")
            
    except Exception as e:
        print(f"Error: {e}")


__main__ - INFO - Embedder: Loading sentence-transformers/all-mpnet-base-v2 on cuda (fp16=True)...
sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


--- Running Embedder Test ---


httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transformers.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/mod

Successfully vectorized 10 sentences.
Sample Embedding (First 3 dims): [0.04150390625, 0.052764892578125, 0.0225982666015625]...


: 

In [ ]:
if RUN_DEBUG_STEPS:
    import time
    import json
    import dataclasses

    print("--- Loading Article Data ---")

    final_result  = NLPResult()
    final_options = NLPOptions(
        min_confidence=0.8,
        max_claims=5,
    )

    print("\n--- Initializing ClaimExtraction Orchestrator ---")
    start_init = time.time()
    pipeline_service = ClaimExtraction(use_gpu=True)
    print(f"Orchestrator ready in {time.time() - start_init:.2f}s")

    print("\n" + "="*80)
    print(f"RUNNING PIPELINE: {article.title}")
    print("="*80)

    try:
        pipeline_service.run(article, final_result, final_options)
    except Exception as e:
        logger.error(f"Pipeline failed at runtime: {e}")

    print("\n" + "="*80)
    print(f"{'FINAL REFINED CLAIMS':^80}")
    print("="*80)

    if not final_result.claims_in_article:
        print("No claims were extracted from this article.")
    else:
        for i, claim in enumerate(final_result.claims_in_article, 1):
            print(f"\nClaim #{i}")
            print(f"Confidence: {claim.confidence:.2f}")
            print(f"Source Sentence Index: {claim.source_sentence_indices}")
            print(f"Text: {claim.decontextualised_claim_text}")
            if claim.NER_entities:
                entity_names = [f"{e.entity_text} ({e.type_of_entity})" for e in claim.NER_entities]
                print(f"Entities: {', '.join(entity_names)}")
            if claim.decontextualised_claim_embedding:
                print(f"Vector: [Stored - {len(claim.decontextualised_claim_embedding)} dimensions]")

    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    print(f"Total Claims Extracted : {len(final_result.claims_in_article)}")
    print(f"Total Unique Entities  : {len(final_result.entities_in_article)}")

    output_file = 'final_claims_output.json'
    with open(output_file, 'w') as f:
        json.dump(dataclasses.asdict(final_result), f, indent=2, default=str)
    print(f"\n✓ Full structured data saved to: {output_file}")


__main__ - INFO - ClaimExtraction: Initializing pipeline orchestrator...
__main__ - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


--- Loading Article Data ---

--- Initializing ClaimExtraction Orchestrator ---


__main__ - INFO - EntityRecognizer: Loading 'dslim/bert-base-NER-uncased' on CUDA (fp16=True)...
httpx - INFO - HTTP Request: HEAD https://huggingface.co/dslim/bert-base-NER-uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dslim/bert-base-NER-uncased/301540b48433e31000058882c971ddd7bc726547/config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/dslim/bert-base-NER-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dslim/bert-base-NER-uncased/301540b48433e31000058882c971ddd7bc726547/tokenizer_config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/dslim/bert-base-NER-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: GET https://hugging